In [1]:
!pip install dgl

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.5/6.5 MB 54.4 MB/s eta 0:00:00


In [2]:
# Complete Fraud Detection System with GNNs
# Based on the working DGL notebook with added requirements
# Includes: RGCN, GAT, Baselines, Explainability, Model Saving

import numpy as np
import pandas as pd
from time import time
import os
import pickle
import json

# DGL and PyTorch imports
os.environ['DGLBACKEND'] = 'pytorch'
import torch
import torch.nn as nn
import torch.nn.functional as F
import dgl
from dgl.nn.pytorch import HeteroGraphConv, HeteroEmbedding, GraphConv, GATConv
from torch.nn import Linear

# Baseline models
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, precision_recall_curve

# Visualization
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import networkx as nx

print("All imports successful!")

# =============================================================================
# DATA LOADING AND PREPROCESSING (Same as working notebook)
# =============================================================================

def load_and_prepare_data():
    """Load and prepare data following the working notebook approach"""
    
    print("Loading IEEE fraud detection dataset...")
    
    # Load datasets
    transactions_df = pd.read_csv('/kaggle/input/ieee-fraud-detection/train_transaction.csv')
    identity_df = pd.read_csv('/kaggle/input/ieee-fraud-detection/train_identity.csv')
    
    num_transactions = transactions_df.shape[0]
    print(f"Number of transactions: {num_transactions}")
    print(f"Proportion of fraudulent transactions: {transactions_df.isFraud.value_counts(normalize=True)[1]:.4f}")
    print(f"Number of transactions with ID data: {identity_df.shape[0]}")
    
    # Train/val split (temporal)
    TRAIN_VAL_RATIO = 0.75
    n_train = int(transactions_df.shape[0] * TRAIN_VAL_RATIO)
    train_ids = transactions_df.TransactionID[:n_train]
    val_ids = transactions_df.TransactionID[n_train:]
    
    # Feature preparation
    id_cols = ['card1','card2','card3','card4','card5','card6','ProductCD',
               'addr1','addr2','P_emaildomain','R_emaildomain']
    cat_cols = ['M1','M2','M3','M4','M5','M6','M7','M8','M9']
    
    # Get features and labels
    transactions_non_features = ['isFraud','TransactionDT'] + id_cols
    features_cols = [col for col in transactions_df.columns if col not in transactions_non_features]
    
    # Create feature dataframe with one-hot encoding
    features_df = pd.get_dummies(transactions_df[features_cols], columns=cat_cols).fillna(0)
    
    # Log transform transaction amount
    features_df['TransactionAmt'] = features_df['TransactionAmt'].apply(np.log10)
    
    # Create labels
    labels_df = transactions_df[['TransactionID','isFraud']]
    
    # Node types for graph
    node_types = id_cols + list(identity_df.columns)
    node_types.remove('TransactionID')
    
    # Join datasets
    full_identity_df = identity_df.merge(
        transactions_df[id_cols + ['TransactionID']], 
        on='TransactionID', how='right'
    )
    
    # Create edge dataframes
    edge_dfs = {}
    for ntype in node_types:
        edge_dfs[ntype] = full_identity_df[['TransactionID', ntype]].dropna()
    
    print(f"Features shape: {features_df.shape}")
    print(f"Training samples: {len(train_ids)}")
    print(f"Validation samples: {len(val_ids)}")
    
    return (transactions_df, identity_df, features_df, labels_df, 
            train_ids, val_ids, node_types, edge_dfs, full_identity_df)

# =============================================================================
# GRAPH CONSTRUCTION (Same as working notebook)
# =============================================================================

def build_heterogeneous_graph(transactions_df, features_df, labels_df, node_types, edge_dfs):
    """Build heterogeneous graph following the working notebook approach"""
    
    print("Building heterogeneous graph...")
    
    # Build ID to node mappings
    id_to_node = {}
    
    # Transaction/target nodes
    id_to_node['target'] = dict([(v,k) for k,v in dict(transactions_df['TransactionID']).items()])
    
    # Other node types
    for ntype in node_types:
        if ntype in edge_dfs and len(edge_dfs[ntype]) > 0:
            new_nodes_ids = edge_dfs[ntype][ntype].unique()
            new_nodes_dgl = np.arange(len(new_nodes_ids))
            id_to_node[ntype] = {a:b for a,b in zip(new_nodes_ids, new_nodes_dgl)}
    
    # Build edge lists
    edgelists = {}
    num_nodes_dict = {}
    
    for ntype in node_types:
        if ntype not in edge_dfs or len(edge_dfs[ntype]) == 0:
            continue
            
        # Edge type triples
        edge_type = ('target', f'target<>{ntype}', ntype)
        rev_edge_type = (ntype, f'{ntype}<>target', 'target')
        
        # Source and destination nodes
        source_nodes = edge_dfs[ntype]['TransactionID'].apply(
            lambda a: id_to_node['target'][a]
        ).to_numpy()
        
        destination_nodes = edge_dfs[ntype][ntype].apply(
            lambda a: id_to_node[ntype][a]
        ).to_numpy()
        
        # Add edges
        edgelists[edge_type] = (source_nodes, destination_nodes)
        edgelists[rev_edge_type] = (destination_nodes, source_nodes)
        
        # Count nodes
        num_nodes_dict[ntype] = len(np.unique(destination_nodes))
    
    # Add self-loops for target nodes
    all_target_nodes = np.arange(len(transactions_df))
    edgelists[('target','target<>target','target')] = (all_target_nodes, all_target_nodes)
    num_nodes_dict['target'] = len(transactions_df)
    
    # Create graph
    g = dgl.heterograph(edgelists, num_nodes_dict)
    
    # Add features
    feature_tensor = torch.from_numpy(features_df.drop('TransactionID', axis=1).to_numpy()).float()
    g.nodes['target'].data['features'] = feature_tensor
    
    # Feature normalization
    mean = torch.mean(g.ndata['features']['target'], axis=0)
    std = torch.sqrt(torch.sum((g.ndata['features']['target'] - mean)**2, axis=0) / 
                     g.ndata['features']['target'].shape[0])
    g.ndata['features']['target'] = (g.ndata['features']['target'] - mean) / std
    
    print(f"Graph created: {g.num_nodes('target')} target nodes")
    print(f"Node types: {len([nt for nt in node_types if nt in num_nodes_dict])}")
    
    return g, id_to_node

# =============================================================================
# ENHANCED MODEL ARCHITECTURES
# =============================================================================

class FFBlock(nn.Module):
    """Feed-forward block for preprocessing/postprocessing"""
    
    def __init__(self, in_dim, hidden_dim, out_dim, n_layers):
        super().__init__()
        self.input_layer = Linear(in_dim, hidden_dim)
        self.hidden_layer = Linear(hidden_dim, hidden_dim)
        self.output_layer = Linear(hidden_dim, out_dim)
        self.n_layers = n_layers
    
    def forward(self, in_feats):
        h = self.input_layer(in_feats)
        h = F.relu(h)
        for i in range(1, self.n_layers):
            h = self.hidden_layer(h)
            h = F.relu(h)
        h = self.output_layer(h)
        return h

class EnhancedRGCN(nn.Module):
    """Enhanced RGCN with explainability features"""
    
    def __init__(self, target_feature_dim, config, graph, node_types):
        super().__init__()
        self.config = config
        self.node_types = [nt for nt in node_types if nt in graph.ntypes]
        
        # Create model dictionaries
        entry_module_dict = {
            etype: GraphConv(config['input_dim'], config['hidden_dim']) 
            for etype in graph.etypes
        }
        
        hidden_module_dict = {
            etype: GraphConv(config['hidden_dim'], config['hidden_dim']) 
            for etype in graph.etypes
        }
        
        final_model_dict1 = {
            etype: GraphConv(config['hidden_dim'], config['target_out_dim']) 
            for src, etype, dst in graph.canonical_etypes if dst == 'target'
        }
        
        final_model_dict2 = {
            etype: GraphConv(config['hidden_dim'], 1) 
            for src, etype, dst in graph.canonical_etypes if dst != 'target'
        }
        
        final_model_dict = {**final_model_dict1, **final_model_dict2}
        
        # Embeddings for non-target nodes
        num_embeddings_dict = {
            src: graph.num_nodes(src) 
            for src, etype, dst in graph.canonical_etypes 
            if dst == 'target' and src != 'target'
        }
        
        if num_embeddings_dict:
            self.embed_layer = HeteroEmbedding(num_embeddings_dict, config['input_dim'])
        else:
            self.embed_layer = None
        
        # Network layers
        self.target_preprocessing = FFBlock(
            target_feature_dim, 
            config['target_preprocessing_hidden_dim'], 
            config['input_dim'],
            config['target_preprocessing_layers']
        )
        
        self.conv1 = HeteroGraphConv(entry_module_dict, aggregate='sum')
        self.conv2 = HeteroGraphConv(hidden_module_dict, aggregate='sum')
        self.conv3 = HeteroGraphConv(final_model_dict, aggregate='sum')
        
        self.target_postprocessing = FFBlock(
            config['target_out_dim'],
            config['target_postprocessing_hidden_dim'],
            1,
            config['target_postprocessing_layers']
        )
        
        self.num_conv_layers = config['conv_layers']
        
    def forward(self, graph, input_features, return_embeddings=False):
        # Create embeddings
        if self.embed_layer is not None and self.node_types:
            embeds = self.embed_layer({
                ntype: graph.nodes(ntype) for ntype in self.node_types
            })
        else:
            embeds = {}
        
        # Preprocess target features
        target_features = self.target_preprocessing(input_features.float())
        embeds['target'] = target_features
        
        # Graph convolutions
        h1 = self.conv1(graph, embeds)
        h1 = {k: F.relu(v) for k, v in h1.items()}
        
        h2 = h1
        for i in range(2, self.num_conv_layers):
            h2 = self.conv2(graph, h2)
            h2 = {k: F.relu(v) for k, v in h2.items()}
        
        h3 = self.conv3(graph, h2)
        
        # Postprocess target output
        if 'target' in h3:
            h3['target'] = self.target_postprocessing(h3['target'])
        
        if return_embeddings:
            return h3, {'layer1': h1, 'layer2': h2, 'layer3': h3}
        
        return h3

class HeteroGAT(nn.Module):
    """Graph Attention Network for heterogeneous graphs"""
    
    def __init__(self, target_feature_dim, config, graph, node_types):
        super().__init__()
        self.config = config
        self.node_types = [nt for nt in node_types if nt in graph.ntypes]
        
        # GAT-specific parameters
        heads = config.get('attention_heads', 4)
        hidden_dim = config['hidden_dim']
        
        # Create GAT dictionaries
        entry_module_dict = {
            etype: GATConv(config['input_dim'], hidden_dim // heads, heads, 
                          feat_drop=0.1, attn_drop=0.1, activation=F.relu)
            for etype in graph.etypes
        }
        
        final_model_dict = {
            etype: GATConv(hidden_dim, config['target_out_dim'] if dst == 'target' else 1, 
                          1, feat_drop=0.1, attn_drop=0.1)
            for src, etype, dst in graph.canonical_etypes
        }
        
        # Embeddings
        num_embeddings_dict = {
            src: graph.num_nodes(src) 
            for src, etype, dst in graph.canonical_etypes 
            if dst == 'target' and src != 'target'
        }
        
        if num_embeddings_dict:
            self.embed_layer = HeteroEmbedding(num_embeddings_dict, config['input_dim'])
        else:
            self.embed_layer = None
        
        # Network components
        self.target_preprocessing = FFBlock(
            target_feature_dim, 
            config['target_preprocessing_hidden_dim'], 
            config['input_dim'],
            config['target_preprocessing_layers']
        )
        
        self.gat1 = HeteroGraphConv(entry_module_dict, aggregate='sum')
        self.gat2 = HeteroGraphConv(final_model_dict, aggregate='sum')
        
        self.target_postprocessing = FFBlock(
            config['target_out_dim'],
            config['target_postprocessing_hidden_dim'],
            1,
            config['target_postprocessing_layers']
        )
    
    def forward(self, graph, input_features, return_attention=False):
        # Create embeddings
        if self.embed_layer is not None and self.node_types:
            embeds = self.embed_layer({
                ntype: graph.nodes(ntype) for ntype in self.node_types
            })
        else:
            embeds = {}
        
        # Preprocess target features
        target_features = self.target_preprocessing(input_features.float())
        embeds['target'] = target_features
        
        # GAT layers
        h1 = self.gat1(graph, embeds)
        h1 = {k: F.relu(v.flatten(1)) if v.dim() > 2 else F.relu(v) for k, v in h1.items()}
        
        h2 = self.gat2(graph, h1)
        
        # Postprocess target output
        if 'target' in h2:
            target_out = h2['target']
            if target_out.dim() > 2:
                target_out = target_out.mean(dim=1)  # Average over attention heads
            h2['target'] = self.target_postprocessing(target_out)
        
        return h2

# =============================================================================
# BASELINE MODELS
# =============================================================================

class BaselineModels:
    """Baseline Random Forest and SVM models"""
    
    def __init__(self):
        self.models = {}
        self.scaler = StandardScaler()
        self.results = {}
        
    def prepare_baseline_data(self, features_df, labels_df, train_ids, val_ids, id_to_node):
        """Prepare data for baseline models"""
        
        # Convert to masks
        train_mask = [id_to_node['target'][x] for x in train_ids]
        val_mask = [id_to_node['target'][x] for x in val_ids]
        
        # Feature matrix (exclude TransactionID)
        feature_cols = [col for col in features_df.columns if col != 'TransactionID']
        X = features_df[feature_cols].values
        y = labels_df['isFraud'].values
        
        # Train/val split
        X_train, X_val = X[train_mask], X[val_mask]
        y_train, y_val = y[train_mask], y[val_mask]
        
        # Scale features
        X_train_scaled = self.scaler.fit_transform(X_train)
        X_val_scaled = self.scaler.transform(X_val)
        
        return X_train_scaled, X_val_scaled, y_train, y_val, feature_cols
        
    def train_random_forest(self, X_train, X_val, y_train, y_val):
        """Train Random Forest model"""
        
        print("Training Random Forest...")
        
        rf = RandomForestClassifier(
            n_estimators=200,
            max_depth=15,
            min_samples_split=10,
            class_weight='balanced',
            random_state=42,
            n_jobs=-1
        )
        
        rf.fit(X_train, y_train)
        
        # Evaluate
        train_score = rf.score(X_train, y_train)
        val_score = rf.score(X_val, y_val)
        
        # Probabilities for AUC
        val_probs = rf.predict_proba(X_val)[:, 1]
        val_auc = roc_auc_score(y_val, val_probs)
        
        self.models['random_forest'] = rf
        self.results['random_forest'] = {
            'train_accuracy': train_score,
            'val_accuracy': val_score,
            'val_auc': val_auc,
            'val_probabilities': val_probs
        }
        
        print(f"Random Forest - Val Accuracy: {val_score:.4f}, Val AUC: {val_auc:.4f}")
        
        return rf
    
    def train_svm(self, X_train, X_val, y_train, y_val):
        """Train SVM model"""
        
        print("Training SVM...")
        
        # Use subset for SVM due to computational constraints
        max_samples = 50000
        if len(X_train) > max_samples:
            indices = np.random.choice(len(X_train), max_samples, replace=False)
            X_train_subset = X_train[indices]
            y_train_subset = y_train[indices]
        else:
            X_train_subset = X_train
            y_train_subset = y_train
        
        svm = SVC(
            probability=True,
            class_weight='balanced',
            random_state=42
        )
        
        svm.fit(X_train_subset, y_train_subset)
        
        # Evaluate
        val_score = svm.score(X_val, y_val)
        val_probs = svm.predict_proba(X_val)[:, 1]
        val_auc = roc_auc_score(y_val, val_probs)
        
        self.models['svm'] = svm
        self.results['svm'] = {
            'val_accuracy': val_score,
            'val_auc': val_auc,
            'val_probabilities': val_probs
        }
        
        print(f"SVM - Val Accuracy: {val_score:.4f}, Val AUC: {val_auc:.4f}")
        
        return svm

# =============================================================================
# TRAINING FUNCTIONS
# =============================================================================

def create_config():
    """Create configuration for models"""
    return {
        'input_dim': 8,
        'hidden_dim': 8,
        'target_out_dim': 8,
        'conv_layers': 2,
        'attention_heads': 4,
        'target_preprocessing_hidden_dim': 32,
        'target_preprocessing_layers': 3,
        'target_postprocessing_hidden_dim': 8,
        'target_postprocessing_layers': 2,
        'learning_rate': 0.001,
        'loss_multiplier': 15,
        'num_epochs': 120
    }

def train_gnn_model(model, graph, labels, train_mask, val_mask, config, model_name):
    """Train GNN model with the working notebook approach"""
    
    print(f"Training {model_name}...")
    
    # Prepare training
    optimizer = torch.optim.Adam(model.parameters(), lr=config['learning_rate'])
    
    # Weighted loss
    train_labels = labels[train_mask]
    weight_vector = (torch.ones(train_labels.shape) + 
                    train_labels * config['loss_multiplier']).reshape((-1, 1))
    
    val_labels = labels[val_mask]
    val_weight_vector = (torch.ones(val_labels.shape) + 
                        val_labels * config['loss_multiplier']).reshape((-1, 1))
    
    loss_fn = torch.nn.BCELoss(weight_vector)
    val_loss_fn = torch.nn.BCELoss(val_weight_vector)
    
    # Training history
    history = {'loss': [], 'val_loss': [], 'val_acc': [], 'val_auc': []}
    best_val_acc = 0
    best_model_state = None
    
    features = graph.nodes['target'].data['features']
    
    for epoch in range(config['num_epochs']):
        # Training step
        model.train()
        optimizer.zero_grad()
        
        logits_dict = model(graph, features)
        logits = logits_dict['target']
        probs = torch.sigmoid(logits)
        preds = (probs > 0.5).float()
        
        # Reshape labels to match predictions
        labels_reshaped = labels.reshape_as(preds)
        
        # Compute losses
        loss = loss_fn(probs[train_mask], labels_reshaped[train_mask])
        
        with torch.no_grad():
            val_loss = val_loss_fn(probs[val_mask], labels_reshaped[val_mask])
            val_acc = (preds[val_mask] == labels_reshaped[val_mask]).float().mean()
            
            # AUC calculation
            try:
                val_auc = roc_auc_score(labels[val_mask].numpy(), 
                                       probs[val_mask].squeeze().numpy())
            except:
                val_auc = 0.5
        
        # Backward pass
        loss.backward()
        optimizer.step()
        
        # Store history
        history['loss'].append(loss.detach().numpy())
        history['val_loss'].append(val_loss.detach().numpy())
        history['val_acc'].append(val_acc.item())
        history['val_auc'].append(val_auc)
        
        # Save best model
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_model_state = model.state_dict().copy()
        
        # Progress reporting
        if epoch % 10 == 9:
            print(f"Epoch {epoch+1}: Loss={loss:.4f}, Val Acc={val_acc:.4f}, Val AUC={val_auc:.4f}")
    
    # Load best model
    if best_model_state is not None:
        model.load_state_dict(best_model_state)
    
    print(f"{model_name} training completed. Best Val Acc: {best_val_acc:.4f}")
    
    return model, history

# =============================================================================
# EXPLAINABILITY FEATURES
# =============================================================================

class ExplainabilityAnalyzer:
    """Analyze model predictions and provide explanations"""
    
    def __init__(self, model, graph, feature_names, id_to_node):
        self.model = model
        self.graph = graph
        self.feature_names = feature_names
        self.id_to_node = id_to_node
        
    def analyze_feature_importance(self, transaction_idx, method='gradient'):
        """Analyze feature importance for a specific transaction"""
        
        self.model.eval()
        
        # Get node features
        node_features = self.graph.nodes['target'].data['features'][transaction_idx:transaction_idx+1]
        node_features.requires_grad_(True)
        
        # Create mini-graph for single transaction
        mini_graph = dgl.graph(([], []), num_nodes=1)
        mini_graph.ndata['features'] = node_features
        
        # Forward pass
        logits = self.model.target_postprocessing(
            self.model.target_preprocessing(node_features)
        )
        prob = torch.sigmoid(logits)
        
        if method == 'gradient':
            # Compute gradients
            prob.backward()
            importance = torch.abs(node_features.grad).squeeze().detach().numpy()
            
            # Get top features
            top_indices = np.argsort(importance)[-10:][::-1]
            
            feature_importance = {}
            for i, idx in enumerate(top_indices):
                if idx < len(self.feature_names):
                    feature_importance[self.feature_names[idx]] = {
                        'importance': float(importance[idx]),
                        'value': float(node_features[0, idx].detach().numpy()),
                        'rank': i + 1
                    }
        
        return feature_importance
    
    def explain_prediction(self, transaction_id):
        """Provide comprehensive explanation for a transaction"""
        
        # Get node index
        if transaction_id not in self.id_to_node['target']:
            return {"error": "Transaction ID not found"}
        
        node_idx = self.id_to_node['target'][transaction_id]
        
        # Get prediction
        self.model.eval()
        with torch.no_grad():
            features = self.graph.nodes['target'].data['features']
            logits = self.model(self.graph, features)['target']
            prob = torch.sigmoid(logits[node_idx]).item()
        
        # Feature importance
        feature_importance = self.analyze_feature_importance(node_idx)
        
        # Generate explanation
        prediction = 'Fraud' if prob > 0.5 else 'Legitimate'
        confidence = max(prob, 1-prob)
        
        explanation_text = f"Transaction {transaction_id} is predicted as {prediction} "
        explanation_text += f"with {confidence:.1%} confidence.\n\n"
        
        if feature_importance:
            explanation_text += "Top contributing features:\n"
            for feature, info in list(feature_importance.items())[:5]:
                explanation_text += f"- {feature}: {info['value']:.3f} (importance: {info['importance']:.3f})\n"
        
        return {
            'transaction_id': transaction_id,
            'fraud_probability': prob,
            'prediction': prediction,
            'confidence': confidence,
            'feature_importance': feature_importance,
            'explanation_text': explanation_text
        }

# =============================================================================
# VISUALIZATION FUNCTIONS
# =============================================================================

def create_training_comparison_plot(gnn_histories, baseline_results):
    """Create comparison plot of all model performances"""
    
    fig = make_subplots(
        rows=2, cols=2,
        subplot_titles=['GNN Training Loss', 'GNN Validation Accuracy', 
                       'GNN Validation AUC', 'Model Comparison'],
        specs=[[{"secondary_y": False}, {"secondary_y": False}],
               [{"secondary_y": False}, {"secondary_y": False}]]
    )
    
    # Plot GNN training curves
    for model_name, history in gnn_histories.items():
        fig.add_trace(
            go.Scatter(y=history['loss'], name=f'{model_name} Loss',
                      line=dict(dash='solid')),
            row=1, col=1
        )
        
        fig.add_trace(
            go.Scatter(y=history['val_acc'], name=f'{model_name} Val Acc'),
            row=1, col=2
        )
        
        fig.add_trace(
            go.Scatter(y=history['val_auc'], name=f'{model_name} Val AUC'),
            row=2, col=1
        )
    
    # Model comparison
    model_names = []
    accuracies = []
    aucs = []
    
    # Add GNN results
    for model_name, history in gnn_histories.items():
        model_names.append(model_name)
        accuracies.append(max(history['val_acc']))
        aucs.append(max(history['val_auc']))
    
    # Add baseline results
    for model_name, results in baseline_results.items():
        model_names.append(model_name.title())
        accuracies.append(results['val_accuracy'])
        aucs.append(results['val_auc'])
    
    fig.add_trace(
        go.Bar(x=model_names, y=accuracies, name='Validation Accuracy',
               marker_color='lightblue'),
        row=2, col=2
    )
    
    fig.add_trace(
        go.Bar(x=model_names, y=aucs, name='Validation AUC',
               marker_color='lightcoral'),
        row=2, col=2
    )
    
    fig.update_layout(height=800, title_text="Model Training Comparison")
    fig.update_xaxes(title_text="Epoch", row=1, col=1)
    fig.update_xaxes(title_text="Epoch", row=1, col=2)
    fig.update_xaxes(title_text="Epoch", row=2, col=1)
    fig.update_xaxes(title_text="Model", row=2, col=2)
    
    return fig

# =============================================================================
# MAIN EXECUTION FUNCTION
# =============================================================================

def main():
    """Main execution function"""
    
    print("=" * 60)
    print("ENHANCED FRAUD DETECTION SYSTEM")
    print("=" * 60)
    
    # 1. Load and prepare data
    data_results = load_and_prepare_data()
    (transactions_df, identity_df, features_df, labels_df, 
     train_ids, val_ids, node_types, edge_dfs, full_identity_df) = data_results
    
    # 2. Build heterogeneous graph
    graph, id_to_node = build_heterogeneous_graph(
        transactions_df, features_df, labels_df, node_types, edge_dfs
    )
    
    # 3. Prepare training data
    train_mask = [id_to_node['target'][x] for x in train_ids]
    val_mask = [id_to_node['target'][x] for x in val_ids]
    labels = torch.tensor(labels_df['isFraud'].to_numpy()).float()
    
    print(f"Training mask size: {len(train_mask)}")
    print(f"Validation mask size: {len(val_mask)}")
    
    # 4. Train baseline models
    print("\n" + "=" * 40)
    print("TRAINING BASELINE MODELS")
    print("=" * 40)
    
    baseline_trainer = BaselineModels()
    baseline_data = baseline_trainer.prepare_baseline_data(
        features_df, labels_df, train_ids, val_ids, id_to_node
    )
    X_train, X_val, y_train, y_val, feature_names = baseline_data
    
    # Train Random Forest
    rf_model = baseline_trainer.train_random_forest(X_train, X_val, y_train, y_val)
    
    # Train SVM
    svm_model = baseline_trainer.train_svm(X_train, X_val, y_train, y_val)
    
    # 5. Train GNN models
    print("\n" + "=" * 40)
    print("TRAINING GNN MODELS")
    print("=" * 40)
    
    config = create_config()
    target_feature_dim = graph.ndata['features']['target'].shape[1]
    
    # Train Enhanced RGCN
    rgcn_model = EnhancedRGCN(target_feature_dim, config, graph, node_types)
    rgcn_model, rgcn_history = train_gnn_model(
        rgcn_model, graph, labels, train_mask, val_mask, config, "Enhanced RGCN"
    )
    
    # Train GAT
    gat_model = HeteroGAT(target_feature_dim, config, graph, node_types)
    gat_model, gat_history = train_gnn_model(
        gat_model, graph, labels, train_mask, val_mask, config, "Hetero GAT"
    )
    
    # 6. Model evaluation and comparison
    print("\n" + "=" * 40)
    print("MODEL EVALUATION")
    print("=" * 40)
    
    # Evaluate GNN models
    gnn_histories = {
        'Enhanced RGCN': rgcn_history,
        'Hetero GAT': gat_history
    }
    
    # Print comparison results
    print("\nFinal Model Performance:")
    print("-" * 50)
    
    for model_name, history in gnn_histories.items():
        best_acc = max(history['val_acc'])
        best_auc = max(history['val_auc'])
        print(f"{model_name:15} | Acc: {best_acc:.4f} | AUC: {best_auc:.4f}")
    
    for model_name, results in baseline_trainer.results.items():
        acc = results['val_accuracy']
        auc = results['val_auc']
        print(f"{model_name.title():15} | Acc: {acc:.4f} | AUC: {auc:.4f}")
    
    # 7. Select best model
    best_gnn_model = rgcn_model
    best_gnn_name = "Enhanced RGCN"
    best_gnn_acc = max(rgcn_history['val_acc'])
    
    if max(gat_history['val_acc']) > best_gnn_acc:
        best_gnn_model = gat_model
        best_gnn_name = "Hetero GAT"
        best_gnn_acc = max(gat_history['val_acc'])
    
    print(f"\nBest GNN Model: {best_gnn_name} (Accuracy: {best_gnn_acc:.4f})")
    
    # 8. Create explainability analyzer
    print("\n" + "=" * 40)
    print("CREATING EXPLAINABILITY FEATURES")
    print("=" * 40)
    
    explainer = ExplainabilityAnalyzer(
        best_gnn_model, graph, feature_names, id_to_node
    )
    
    # Test explanation on a sample transaction
    sample_transaction_id = transactions_df.iloc[val_mask[0]]['TransactionID']
    sample_explanation = explainer.explain_prediction(sample_transaction_id)
    
    print("Sample Explanation:")
    print("-" * 30)
    print(sample_explanation['explanation_text'])
    
    # 9. Create visualizations
    print("\n" + "=" * 40)
    print("CREATING VISUALIZATIONS")
    print("=" * 40)
    
    # Training comparison plot
    comparison_plot = create_training_comparison_plot(gnn_histories, baseline_trainer.results)
    comparison_plot.write_html("model_comparison.html")
    print("Model comparison plot saved to: model_comparison.html")
    
    # 10. Save all models and results
    print("\n" + "=" * 40)
    print("SAVING MODELS AND RESULTS")
    print("=" * 40)
    
    # Save the complete system
    save_dict = {
        'best_gnn_model_state': best_gnn_model.state_dict(),
        'best_gnn_model_type': best_gnn_name,
        'rgcn_model_state': rgcn_model.state_dict(),
        'gat_model_state': gat_model.state_dict(),
        'baseline_models': {
            'random_forest': baseline_trainer.models['random_forest'],
            'svm': baseline_trainer.models['svm'],
            'scaler': baseline_trainer.scaler
        },
        'config': config,
        'graph_info': {
            'num_nodes': graph.num_nodes('target'),
            'node_types': [nt for nt in node_types if nt in graph.ntypes],
            'num_edges': sum([graph.num_edges(etype) for etype in graph.etypes])
        },
        'data_info': {
            'feature_names': feature_names,
            'id_to_node': id_to_node,
            'train_mask': train_mask,
            'val_mask': val_mask
        },
        'results': {
            'gnn_histories': gnn_histories,
            'baseline_results': baseline_trainer.results,
            'best_model_performance': {
                'model_name': best_gnn_name,
                'accuracy': best_gnn_acc,
                'auc': max(gnn_histories[best_gnn_name]['val_auc'])
            }
        },
        'target_feature_dim': target_feature_dim
    }
    
    # Save with pickle for complex objects
    with open('fraud_detection_complete_system.pkl', 'wb') as f:
        pickle.dump(save_dict, f)
    
    # Save model states with torch
    torch.save({
        'best_gnn_model_state': best_gnn_model.state_dict(),
        'rgcn_model_state': rgcn_model.state_dict(),
        'gat_model_state': gat_model.state_dict(),
        'config': config,
        'target_feature_dim': target_feature_dim,
        'best_model_name': best_gnn_name
    }, 'fraud_detection_models.pth')
    
    # Save configuration and results as JSON
    json_results = {
        'model_performance': {
            'Enhanced_RGCN': {
                'best_accuracy': float(max(rgcn_history['val_acc'])),
                'best_auc': float(max(rgcn_history['val_auc'])),
                'final_loss': float(rgcn_history['loss'][-1])
            },
            'Hetero_GAT': {
                'best_accuracy': float(max(gat_history['val_acc'])),
                'best_auc': float(max(gat_history['val_auc'])),
                'final_loss': float(gat_history['loss'][-1])
            },
            'Random_Forest': {
                'accuracy': float(baseline_trainer.results['random_forest']['val_accuracy']),
                'auc': float(baseline_trainer.results['random_forest']['val_auc'])
            },
            'SVM': {
                'accuracy': float(baseline_trainer.results['svm']['val_accuracy']),
                'auc': float(baseline_trainer.results['svm']['val_auc'])
            }
        },
        'best_model': best_gnn_name,
        'dataset_info': {
            'total_transactions': len(transactions_df),
            'fraud_rate': float(transactions_df.isFraud.mean()),
            'training_samples': len(train_mask),
            'validation_samples': len(val_mask),
            'num_features': len(feature_names)
        },
        'config': config
    }
    
    with open('fraud_detection_results.json', 'w') as f:
        json.dump(json_results, f, indent=2)
    
    print("Complete system saved to:")
    print("- fraud_detection_complete_system.pkl (complete system)")
    print("- fraud_detection_models.pth (PyTorch models)")
    print("- fraud_detection_results.json (results summary)")
    
    # 11. Print final summary
    print("\n" + "=" * 60)
    print("TRAINING COMPLETED SUCCESSFULLY!")
    print("=" * 60)
    print(f"Best Model: {best_gnn_name}")
    print(f"Best Accuracy: {best_gnn_acc:.4f}")
    print(f"Best AUC: {max(gnn_histories[best_gnn_name]['val_auc']):.4f}")
    print(f"Total node types in graph: {len([nt for nt in node_types if nt in graph.ntypes])}")
    print(f"Total graph edges: {sum([graph.num_edges(etype) for etype in graph.etypes])}")
    
    print("\nNext steps:")
    print("1. Use the saved models for inference")
    print("2. Implement dashboard with saved system")
    print("3. Deploy for real-time fraud detection")
    
    return {
        'best_model': best_gnn_model,
        'explainer': explainer,
        'graph': graph,
        'results': save_dict,
        'baseline_trainer': baseline_trainer
    }

# =============================================================================
# UTILITY FUNCTIONS FOR MODEL LOADING
# =============================================================================

def load_saved_system(file_path='fraud_detection_complete_system.pkl'):
    """Load the complete saved system"""
    
    with open(file_path, 'rb') as f:
        save_dict = pickle.load(f)
    
    print("System loaded successfully!")
    print(f"Best model: {save_dict['results']['best_model_performance']['model_name']}")
    print(f"Best accuracy: {save_dict['results']['best_model_performance']['accuracy']:.4f}")
    
    return save_dict

def create_model_from_saved(save_dict, model_type='best'):
    """Recreate model from saved state"""
    
    config = save_dict['config']
    target_feature_dim = save_dict['target_feature_dim']
    
    # This would need the graph to be recreated - for inference implementation
    # Will be implemented in the inference notebook
    
    pass

# =============================================================================
# RUN THE COMPLETE SYSTEM
# =============================================================================

if __name__ == "__main__":
    print("Starting Complete Fraud Detection System Training...")
    print("This will train GNN models, baselines, and create explainability features.")
    print("Expected runtime: 15-20 minutes")
    print()
    
    try:
        results = main()
        print("\n🎉 All training completed successfully!")
        
    except Exception as e:
        print(f"\n❌ Error during training: {str(e)}")
        import traceback
        traceback.print_exc()

All imports successful!
Starting Complete Fraud Detection System Training...
This will train GNN models, baselines, and create explainability features.
Expected runtime: 15-20 minutes

ENHANCED FRAUD DETECTION SYSTEM
Loading IEEE fraud detection dataset...
Number of transactions: 590540
Proportion of fraudulent transactions: 0.0350
Number of transactions with ID data: 144233
Features shape: (590540, 391)
Training samples: 442905
Validation samples: 147635
Building heterogeneous graph...
Graph created: 590540 target nodes
Node types: 51
Training mask size: 442905
Validation mask size: 147635

TRAINING BASELINE MODELS
Training Random Forest...
Random Forest - Val Accuracy: 0.9213, Val AUC: 0.8567
Training SVM...
SVM - Val Accuracy: 0.9035, Val AUC: 0.8002

TRAINING GNN MODELS
Training Enhanced RGCN...
Epoch 10: Loss=1.0475, Val Acc=0.8414, Val AUC=0.6383
Epoch 20: Loss=0.9411, Val Acc=0.8699, Val AUC=0.7099
Epoch 30: Loss=0.8930, Val Acc=0.8821, Val AUC=0.7351
Epoch 40: Loss=0.8755, Val 